# Opsly Hybrid Daily Health Check
Valida salud diaria de arquitectura hibrida: VPS control plane + worker compute plane.

## Configuracion

In [ ]:
set -euo pipefail
VPS_SSH_HOST="vps-dragon"
VPS_IP="100.120.151.91"
WORKER_SSH_HOST="opslyquantum@100.80.41.29"
WORKER_IP="100.80.41.29"
PUBLIC_API_HEALTH_URL="https://api.ops.smiletripcare.com/api/health"
echo "Config OK: VPS=$VPS_SSH_HOST ($VPS_IP), WORKER=$WORKER_SSH_HOST ($WORKER_IP)"

## Check 1: VPS control plane

In [ ]:
set -euo pipefail
echo "[1/5] API publica"
curl -sS --max-time 10 "$PUBLIC_API_HEALTH_URL"
echo "\n[2/5] Orchestrator health"
ssh "$VPS_SSH_HOST" "docker exec opsly_orchestrator node -e \"fetch('http://127.0.0.1:3011/health').then(r=>r.text()).then(t=>console.log(t)).catch(()=>process.exit(1))\""
echo "[3/5] LLM gateway health"
ssh "$VPS_SSH_HOST" "curl -sS --max-time 8 http://127.0.0.1:3010/health || true"
echo "[4/5] Redis ping"
ssh "$VPS_SSH_HOST" "REDIS_CONT=\$(docker ps --format '{{.Names}}' | rg 'redis' | head -1); docker exec \"\$REDIS_CONT\" redis-cli ping"
echo "[5/5] Containers clave"
ssh "$VPS_SSH_HOST" "docker ps --format 'table {{.Names}}\t{{.Status}}' | rg 'opsly_orchestrator|opsly_llm_gateway|infra-redis-1|infra-app' || true"

## Check 2: Conectividad VPS -> Worker (Ollama)

In [ ]:
set -euo pipefail
echo "VPS -> Worker Ollama /api/tags"
ssh "$VPS_SSH_HOST" "curl -sS --max-time 8 http://$WORKER_IP:11434/api/tags | python3 -m json.tool | sed -n '1,20p'"

## Check 3: Worker compute plane

In [ ]:
set -euo pipefail
echo "Worker containers"
ssh "$WORKER_SSH_HOST" "docker ps --format 'table {{.Names}}\t{{.Status}}'"
echo "Worker -> VPS Redis y Gateway"
ssh "$WORKER_SSH_HOST" "nc -vz $VPS_IP 6379 && nc -vz $VPS_IP 3010"
echo "Worker Ollama local"
ssh "$WORKER_SSH_HOST" "curl -sS --max-time 5 http://127.0.0.1:11434/api/tags | python3 -m json.tool | sed -n '1,20p'"

## Check 4: Smoke job (openclaw -> worker)
Encola un job `ollama` y consulta estado.

In [ ]:
set -euo pipefail
JOB_JSON=$(ssh "$VPS_SSH_HOST" 'docker exec opsly_orchestrator node -e "(async()=>{const token=process.env.PLATFORM_ADMIN_TOKEN||\"\"; const res=await fetch(\"http://127.0.0.1:3011/internal/enqueue-ollama\",{method:\"POST\",headers:{\"Authorization\":\"Bearer \"+token,\"Content-Type\":\"application/json\"},body:JSON.stringify({tenant_slug:\"platform\",prompt:\"daily health smoke\",task_type:\"summarize\"})}); console.log(await res.text());})().catch(e=>{console.error(e);process.exit(1);});"')
echo "$JOB_JSON"
JOB_ID=$(python3 - <<'PY'
import json,sys
obj=json.loads(sys.stdin.read())
print(obj.get('job_id',''))
PY
<<< "$JOB_JSON")
test -n "$JOB_ID"
echo "Polling job_id=$JOB_ID"
sleep 3
ssh "$VPS_SSH_HOST" "docker exec opsly_orchestrator node -e \"(async()=>{const token=process.env.PLATFORM_ADMIN_TOKEN||''; const r=await fetch('http://127.0.0.1:3011/internal/openclaw-job?job_id=$JOB_ID',{headers:{Authorization:'Bearer '+token}}); console.log(await r.text());})().catch(()=>process.exit(1));\""

## Operacion correctiva (opcional)
Usar solo si un check falla.

In [ ]:
set -euo pipefail
echo "Reinicio rapido servicios core en VPS"
ssh "$VPS_SSH_HOST" "cd /opt/opsly/infra && docker compose --env-file /opt/opsly/.env -f docker-compose.platform.yml up -d llm-gateway orchestrator"
echo "Reinicio worker primary en Mac2011"
ssh "$WORKER_SSH_HOST" "cd ~/opsly/infra && docker compose -f docker-compose.workers.yml up -d worker-primary || true"